# 5 — Population ratios across timepoints and treatments

Stacked bars of cell-type composition across the 5 `(timepoint, treatment)` combinations and per-participant breakdowns. Two variants per figure: all cells / confident only (`scanvi_confidence ≥ 0.5`).

Operates on the thin obs DataFrame from `combine_predictions_obs`.

In [ ]:
import sys
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import CELLASSIGN_DIR, FIGURES_DIR, INTEGRATION_DIR
from src.palette import celltype_palette, normalize_celltype_name
from src.transfer_learning import combine_predictions_obs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

QUERY_KEYS = ['D2_DZ', 'D2_Lapa', 'D4_DZ', 'D4_Lapa']
REFERENCE_H5AD = INTEGRATION_DIR / 'scarches_model' / 'reference_with_latent.h5ad'
QUERY_PATHS = [CELLASSIGN_DIR / f'{k.lower()}_scanvi_predictions.h5ad' for k in QUERY_KEYS]

fig_dir = FIGURES_DIR / 'd2-d4-integrated' / 'population-ratios'
fig_dir.mkdir(parents=True, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.5

TIMEPOINT_ORDER = ['G2D2', 'G2D4', 'G2D10']  # falls back to alphabetical if names differ
TREATMENT_ORDER = ['Dz', 'Lapa']

In [ ]:
df = combine_predictions_obs(
    query_h5ad_paths=QUERY_PATHS,
    reference_h5ad_path=REFERENCE_H5AD,
)
df['Time_point'] = df['Time_point'].astype(str)
df['Treatment']  = df['Treatment'].astype(str)

# Reference doesn't have scanvi_prediction; backfill from scanvi_prediction_ref or initial_cellassign_prediction
if 'scanvi_prediction' not in df.columns:
    df['scanvi_prediction'] = np.nan
ref_mask = df['dataset'] == 'D10_Lapa'
if 'scanvi_prediction_ref' in df.columns:
    df.loc[ref_mask, 'scanvi_prediction'] = df.loc[ref_mask, 'scanvi_prediction_ref']
if 'initial_cellassign_prediction' in df.columns:
    fill = df.loc[ref_mask & df['scanvi_prediction'].isna(), 'initial_cellassign_prediction']
    df.loc[fill.index, 'scanvi_prediction'] = fill

df['scanvi_prediction'] = df['scanvi_prediction'].astype(str).map(normalize_celltype_name)
df.shape, df['dataset'].value_counts(), df['scanvi_prediction'].value_counts().head(10)

In [ ]:
def stacked_composition(df_in: pd.DataFrame, group_col: str, label_col: str = 'scanvi_prediction') -> pd.DataFrame:
    counts = df_in.groupby([group_col, label_col]).size().unstack(fill_value=0)
    return counts.div(counts.sum(axis=1), axis=0)

def plot_stacked(comp_df: pd.DataFrame, ax, title: str):
    cols = list(comp_df.columns)
    colors = [celltype_palette.get(c, '#808080') for c in cols]
    comp_df.plot.bar(stacked=True, color=colors, ax=ax, width=0.85, legend=False)
    ax.set_title(title); ax.set_ylabel('fraction')
    ax.set_ylim(0, 1)
    for tick in ax.get_xticklabels():
        tick.set_rotation(30); tick.set_ha('right')

In [ ]:
# Group key for the 5 (timepoint, treatment) combinations
df['tp_tx'] = df['Time_point'].astype(str) + ' / ' + df['Treatment'].astype(str)

# Order x-axis: D2 < D4 < D10, Dz < Lapa
order = sorted(
    df['tp_tx'].unique(),
    key=lambda s: (s.split(' / ')[0], s.split(' / ')[1]),
)

for variant in ('all', 'confident'):
    if variant == 'all':
        sub = df.copy()
        suffix = 'all_cells'
    else:
        sub = df[df['scanvi_confidence'].fillna(1.0) >= CONFIDENCE_THRESHOLD].copy()
        suffix = 'confident_only'
    comp = stacked_composition(sub, 'tp_tx').reindex(order)
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_stacked(comp, ax, title=f'cell-type composition by (Time_point / Treatment) — {variant}')
    handles = [plt.Rectangle((0,0),1,1, color=celltype_palette.get(c, '#808080')) for c in comp.columns]
    ax.legend(handles, list(comp.columns), bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(fig_dir / f'composition_by_tp_tx_{suffix}.pdf')
    plt.show()

In [ ]:
# Per-participant composition (controls for donor effects)
for variant in ('all', 'confident'):
    sub = df.copy() if variant == 'all' else df[df['scanvi_confidence'].fillna(1.0) >= CONFIDENCE_THRESHOLD].copy()
    sub = sub.dropna(subset=['participant'])
    sub['donor_x_tp_tx'] = sub['participant'].astype(str) + ' | ' + sub['tp_tx']
    comp = stacked_composition(sub, 'donor_x_tp_tx')
    fig, ax = plt.subplots(figsize=(max(10, 0.4 * len(comp)), 5))
    plot_stacked(comp, ax, title=f'per-participant composition — {variant}')
    handles = [plt.Rectangle((0,0),1,1, color=celltype_palette.get(c, '#808080')) for c in comp.columns]
    ax.legend(handles, list(comp.columns), bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig(fig_dir / f'composition_by_donor_{variant}.pdf')
    plt.show()

In [ ]:
# ISC fraction trajectory — sanity check (should be non-increasing across D2 -> D4 -> D10 in matched treatment)
trend = (
    df.assign(is_isc=(df['scanvi_prediction'] == 'ISCs').astype(int))
      .groupby(['Time_point', 'Treatment'])['is_isc']
      .mean()
      .unstack('Treatment')
)
trend